<a href="https://colab.research.google.com/github/gitmystuff/DTSC4050/blob/main/07_AB_Tests/Your_Name_AB_Testing_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A/B Testing: Applied Assignment

**Estimated time: ~60 minutes**

This assignment follows directly from lecture. You'll work through the full A/B testing
pipeline on a single scenario: computing a confidence interval, running a power analysis,
choosing the right test, executing it, and writing a real conclusion.

**Instructions**
* Replace every `# TODO` and `___` with your own code or answer.
* Do not delete the `assert` / sanity-check cells — they're there to help you catch mistakes early.
* Answer every markdown question in the cell provided (double-click to edit).
* Run cells top to bottom. If something breaks, it's usually because an earlier `# TODO` wasn't finished.

**Time budget (rough guide, not enforced)**
| Section | Topic | Minutes |
|---|---|---|
| 1 | Setup | 2 |
| 2 | Confidence Interval for a Single Proportion | 10 |
| 3 | Choosing a Test: Chi-Squared vs. Two-Proportion Z-Test | 8 |
| 4 | Power Analysis & Sample Size | 12 |
| 5 | Running the A/B Test | 15 |
| 6 | Interpreting Results | 10 |
| 7 | Reflection | 3 |


## Section 1: Setup

In [ ]:
# TODO: import the libraries you'll need for this assignment.
# You will need: numpy, pandas, scipy.stats, and statsmodels.stats.api / statsmodels.stats.proportion

# TODO


## Section 2: Confidence Interval for a Single Proportion

A product page currently gets clicked by **62 out of 400** visitors who see it (a "click-through").

Recall from lecture:

* `phat = x / n`
* Normal approximation valid when `n * phat > 5` and `n * (1 - phat) > 5`
* `SE = sqrt(phat * (1 - phat) / n)`
* 95% CI: `phat ± 1.96 * SE`


In [ ]:
x = 62      # successes
n = 400     # trials

# TODO: compute phat
phat = ___

# TODO: check the normal approximation condition. Print True/False for each condition.
condition_1 = ___  # n * phat > 5
condition_2 = ___  # n * (1 - phat) > 5
print(condition_1, condition_2)

# TODO: compute the standard error
se = ___

# TODO: compute the 95% confidence interval (lower, upper)
z = 1.96
lower = ___
upper = ___

print(f'phat: {phat:.4f}')
print(f'SE: {se:.4f}')
print(f'95% CI: [{lower:.4f}, {upper:.4f}]')


In [ ]:
# Sanity check -- do not edit this cell.
assert round(phat, 4) == 0.155, "Check your phat calculation."
assert round(se, 4) == 0.0181, "Check your standard error calculation."
print("Looks good!")


**Question:** Suppose instead you only had 20 visitors, and 3 of them clicked. Would the
normal approximation still be valid? Why or why not, and what would you do instead?

*Your answer here:*


## Section 3: Choosing a Test

For each scenario below, state which test you would use — the **two-proportion z-test** or
the **chi-squared test** — and give one sentence of justification. (Hint: think about how many
groups/categories are involved, and whether you need a confidence interval or direction.)

1. You're comparing conversion rate between **Control** and **Treatment** in a standard A/B test, and you want to report a confidence interval on the lift.

   *Your answer here:*

2. You're running an A/B/C test comparing three different checkout button colors, and you just want to know if conversion rate differs across the three at all.

   *Your answer here:*

3. You want to verify there wasn't a **Sample Ratio Mismatch** — that your 50/50 traffic split actually resulted in roughly equal group sizes.

   *Your answer here:*

4. You're comparing click-through rate between Control and Treatment and you specifically want to know if Treatment is *better*, not just *different*.

   *Your answer here:*


## Section 4: Power Analysis & Sample Size

Your company's landing page currently converts at **18%**. The design team believes a
redesign can push that to **21%**. You've been asked to size an A/B test with:

* Significance level (alpha): 0.05
* Power: 0.80
* Equal group sizes (ratio = 1)


In [ ]:
import statsmodels.stats.api as sms

baseline = 0.18
target = 0.21

# TODO: compute the effect size (Cohen's h) using sms.proportion_effectsize
effect_size = ___

# TODO: solve for the required sample size per group using sms.NormalIndPower().solve_power
sample_size = ___
sample_size = int(sample_size)

print(f'Effect size: {effect_size:.4f}')
print(f'Required sample size per group: {sample_size}')


**Question:** If your marketing team says they can only get 1,500 users per group in the
time available, what are your options? List at least two.

*Your answer here:*


## Section 5: Running the A/B Test

Below is a small simulated dataset for a "Learn More" button redesign. `group` is `control` or
`treatment`, and `converted` is 1 if the visitor clicked, 0 if not.

Run the cell to generate the data, then complete the analysis.


In [ ]:
import numpy as np
import pandas as pd

np.random.seed(7)

n_per_group = 2000
control = np.random.binomial(1, 0.18, n_per_group)
treatment = np.random.binomial(1, 0.208, n_per_group)

df = pd.DataFrame({
    'group': ['control'] * n_per_group + ['treatment'] * n_per_group,
    'converted': np.concatenate([control, treatment])
})

df.sample(5, random_state=1)


In [ ]:
# TODO: compute the conversion rate, standard deviation, and standard error for each group.
# Hint: use df.groupby('group')['converted'] and .agg([...])

conversion_rates = ___
conversion_rates


In [ ]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

control_group = df[df['group'] == 'control']['converted']
treatment_group = df[df['group'] == 'treatment']['converted']

# TODO: get n and number of successes for each group
n_con = ___
n_treat = ___
successes = [___, ___]
n_obs = [n_con, n_treat]

# TODO: run the two-proportion z-test. Store the p-value in `pval`.
_, pval = ___

# TODO: compute 95% confidence intervals for both groups using proportion_confint
(lower_con, lower_treat), (upper_con, upper_treat) = ___

print(f'p-value: {pval:.4f}')
print(f'Control 95% CI: [{lower_con:.4f}, {upper_con:.4f}]')
print(f'Treatment 95% CI: [{lower_treat:.4f}, {upper_treat:.4f}]')


In [ ]:
# Sanity check -- do not edit this cell.
assert n_con == 2000 and n_treat == 2000, "n_con and n_treat should each be 2000."
assert 0 <= pval <= 1, "pval should be a valid probability -- check your proportions_ztest call."
assert lower_con < upper_con and lower_treat < upper_treat, "Check your proportion_confint call -- lower bound should be less than upper bound."
print("Looks good!")


## Section 6: Interpreting Results

Answer using your actual output from Section 5 -- not made-up numbers.

1. Is the p-value below alpha = 0.05? What does that tell you about the null hypothesis?

   *Your answer here:*

2. The design team's original goal was to push conversion from 18% to at least 20.8%.
   Does the treatment group's confidence interval support that the redesign reached this goal?
   Is this result statistically significant, practically significant, both, or neither?

   *Your answer here:*

3. Would your recommendation change if this test had only run with 200 users per group
   instead of 2,000? Why?

   *Your answer here:*


## Section 7: Reflection

In 2-3 sentences: what is one thing about A/B testing that seemed straightforward in the
lecture material, but felt trickier once you had to apply it in code here?

*Your answer here:*


---
**Submission:** Save this notebook with your answers and TODOs completed, and submit the
`.ipynb` file as instructed by your course platform.
